In [14]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_mcp_adapters.client import MultiServerMCPClient
from pprint import pprint
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver 
from langchain.messages import HumanMessage
from langgraph.graph.state import RunnableConfig

load_dotenv()

True

In [4]:
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=1.0, verbose=True)

In [25]:
client = MultiServerMCPClient(
    {
        "travel_server": {
            "transport": "streamable_http",
            "url": "https://mcp.kiwi.com"
        }
    }
)

In [26]:
tools = await client.get_tools()

In [29]:
pprint(tools)

[StructuredTool(name='search-flight', description='\n# Search for a flight\n\n## Description\n\nUses the Kiwi API to search for available flights between two locations on a specific date.\n\n## How it works\n\nThe tool will:\n1. Search for matching locations to resolve airport codes\n2. Find available flights for the specified route and date range\n\n## Method\n\nCall this tool whenever a user wants to search for flights, regardless of whether they provided exact airport codes or just city names.\n\nYou should display the returned results in a markdown table format: Group the results by price (those who are the cheapest), duration (those who are the shortest, i.e. have the smallest \'totalDurationInSeconds\') and the rest (those that could still be interesting).\n\nAlways display for each flight in order:\n  - In the 1st column: The departure and arrival airports, including layovers (e.g. "Paris CDG → Barcelona BCN → Lisbon LIS")\n  - In the 2nd column: The departure and arrival dates 

In [ ]:
agent = create_agent(
    model=model, 
    tools=tools, 
    checkpointer=InMemorySaver(),
    system_prompt="You are a travel agent. You are given a list of tools to help you find the best flight for a user."
)



In [ ]:
config: RunnableConfig = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Get me a flight from Bentonville (XNA) to Hyderabad (HYD) on 14th of April 2026")]},
    config
)

In [ ]:
pprint(response["messages"][-1].content[0]["text"])

In [5]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__

In [10]:
mcpClient = MultiServerMCPClient(
        {
            "math": {
                "transport": "stdio",  # Local subprocess communication
                "command": "python",
                # Absolute path to your math_server.py file
                "args": ["resources/mcp_server.py"],
            }
        }
    )

In [11]:
mcp_tools = await mcpClient.get_tools()

In [12]:
pprint(mcp_tools)

[StructuredTool(name='get_weather', description='Get weather information for a location.', args_schema={'additionalProperties': False, 'properties': {'location': {'type': 'string'}}, 'required': ['location'], 'type': 'object'}, metadata={'_meta': {'fastmcp': {'tags': []}}}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x00000252CC3F6CA0>)]


In [15]:
mcp_agent = create_agent(
    model=model, 
    tools=mcp_tools, 
    checkpointer=InMemorySaver(),
    system_prompt="You are a weather agent. You are given a list of tools to help you find the weather for a location."
)


In [ ]:
config: RunnableConfig = {"configurable": {"thread_id": "2"}}
mcp_response = await mcp_agent.ainvoke(
    {"messages": [HumanMessage(content="What is the weather in Hyderabad?")]},
    config
)
pprint(mcp_response["messages"][-1].content[0]["text"])